# CellSeg1 CGH P2 Live Training

Run this notebook on the GPU cluster after cloning/pulling the repo. It trains a CellSeg1/SAM LoRA instance-segmentation model on the CGH PA P2 cell-boundary masks copied from the T9 YOLO training export.

The workbook is intentionally separate from the YOLO workbook:

- CellSeg1 trains one untyped positive instance class: cell boundary.
- Clear vs compact labels are preserved in metadata but are not separate CellSeg1 output classes.
- The notebook uses all packaged training tiles unless `CELLSEG1_TRAIN_IDS` is set.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime

import numpy as np
from PIL import Image

CWD = Path.cwd().resolve()
if (CWD / 'training').exists() and (CWD / 'training_data').exists():
    REPO_ROOT = CWD
elif (CWD.parent / 'training').exists() and (CWD.parent / 'training_data').exists():
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError(f'Run this notebook from the repo root or training/: {CWD}')

OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'cellseg1_cluster_live'
RUNTIME_DATASET = OUTPUT_ROOT / 'datasets' / 'cellseg1_instance_train'
PACKAGE_DATASET = REPO_ROOT / 'training_data' / 'dataset' / 'cellseg1_instance_train'
T9_DATASET = Path('/Volumes/T9/CGH_PA_annotation_1/training_data/cellseg1_cgh_p2')
REFERENCE_MODEL_DIR = REPO_ROOT / 'training_data' / 'reference_models'
REFERENCE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.getenv('CELLSEG1_RUN_NAME') or f"cellseg1_cgh_p2_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

USE_T9 = os.getenv('CELLSEG1_USE_T9', '1') != '0' and T9_DATASET.exists()
SOURCE_DATASET = T9_DATASET if USE_T9 else PACKAGE_DATASET
if not SOURCE_DATASET.exists():
    raise FileNotFoundError(f'No CellSeg1 dataset found. Checked: {SOURCE_DATASET}')

print('REPO_ROOT:', REPO_ROOT)
print('SOURCE_DATASET:', SOURCE_DATASET)
print('RUNTIME_DATASET:', RUNTIME_DATASET)
print('RUN_DIR:', RUN_DIR)


In [ ]:
def copy_clean_tree(src: Path, dst: Path, suffixes=('.png', '.csv', '.json', '.md', '.txt')):
    dst.mkdir(parents=True, exist_ok=True)
    copied = 0
    for path in sorted(src.iterdir()):
        if path.name.startswith('._') or path.name == '.DS_Store':
            continue
        if path.is_file() and path.suffix.lower() in suffixes:
            shutil.copy2(path, dst / path.name)
            copied += 1
    return copied

if RUNTIME_DATASET.exists():
    shutil.rmtree(RUNTIME_DATASET)
RUNTIME_DATASET.mkdir(parents=True, exist_ok=True)

if (SOURCE_DATASET / 'train' / 'images').exists():
    image_src = SOURCE_DATASET / 'train' / 'images'
    mask_src = SOURCE_DATASET / 'train' / 'masks'
    metadata_src = SOURCE_DATASET
else:
    image_src = SOURCE_DATASET / 'images'
    mask_src = SOURCE_DATASET / 'masks'
    metadata_src = SOURCE_DATASET / 'metadata'

copy_clean_tree(image_src, RUNTIME_DATASET / 'images', suffixes=('.png',))
copy_clean_tree(mask_src, RUNTIME_DATASET / 'masks', suffixes=('.png',))

for subdir in ['metadata', 'auxiliary_masks', 'semantic_masks', 'previews']:
    src = metadata_src / subdir if subdir == 'metadata' else SOURCE_DATASET / subdir
    if src.exists():
        copy_clean_tree(src, RUNTIME_DATASET / subdir)

# T9 keeps metadata files at dataset root, while the portable package keeps them under metadata/.
if (metadata_src / 'dataset_manifest.csv').exists():
    meta_dst = RUNTIME_DATASET / 'metadata'
    meta_dst.mkdir(exist_ok=True)
    for name in ['dataset_manifest.csv', 'cell_instances.csv', 'boundary_qc.csv']:
        src = metadata_src / name
        if src.exists() and not src.name.startswith('._'):
            shutil.copy2(src, meta_dst / src.name)

print('runtime images:', len(list((RUNTIME_DATASET / 'images').glob('*.png'))))
print('runtime masks:', len(list((RUNTIME_DATASET / 'masks').glob('*.png'))))
print('runtime metadata:', sorted(p.name for p in (RUNTIME_DATASET / 'metadata').glob('*')) if (RUNTIME_DATASET / 'metadata').exists() else [])


In [ ]:
manifest_path = RUNTIME_DATASET / 'metadata' / 'dataset_manifest.csv'
manifest_rows = list(csv.DictReader(manifest_path.open())) if manifest_path.exists() else []

image_files = sorted(p for p in (RUNTIME_DATASET / 'images').glob('*.png') if not p.name.startswith('._'))
mask_files = sorted(p for p in (RUNTIME_DATASET / 'masks').glob('*.png') if not p.name.startswith('._'))
assert image_files, 'No runtime images found.'
assert len(image_files) == len(mask_files), (len(image_files), len(mask_files))
assert [p.stem for p in image_files] == [p.stem for p in mask_files]

qc_rows = []
for image_path, mask_path in zip(image_files, mask_files):
    with Image.open(image_path) as image:
        image_size = image.size
    mask = np.asarray(Image.open(mask_path))
    labels = np.unique(mask)
    labels = labels[labels != 0]
    expected = list(range(1, int(labels.max()) + 1)) if labels.size else []
    actual = [int(x) for x in labels]
    qc_rows.append({
        'tile_id': image_path.stem,
        'image_size': image_size,
        'mask_shape': mask.shape,
        'instances': len(actual),
        'max_label': max(actual) if actual else 0,
        'labels_contiguous': actual == expected,
    })

summary = {
    'source_dataset': str(SOURCE_DATASET),
    'runtime_dataset': str(RUNTIME_DATASET),
    'image_count': len(image_files),
    'mask_count': len(mask_files),
    'manifest_rows': len(manifest_rows),
    'trainable_instances_from_masks': sum(row['instances'] for row in qc_rows),
    'manifest_trainable_instances': sum(int(r['trainable_instances']) for r in manifest_rows) if manifest_rows else None,
    'manifest_clear_trainable': sum(int(r['clear_trainable']) for r in manifest_rows) if manifest_rows else None,
    'manifest_compact_trainable': sum(int(r['compact_trainable']) for r in manifest_rows) if manifest_rows else None,
    'manifest_nuclei_in_tile': sum(int(r['nuclei_in_tile']) for r in manifest_rows) if manifest_rows else None,
    'qc': qc_rows,
}
(RUN_DIR / 'dataset_runtime_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps({k: v for k, v in summary.items() if k != 'qc'}, indent=2))
assert all(row['labels_contiguous'] for row in qc_rows), 'One or more masks have non-contiguous labels.'


In [ ]:
def nvidia_snapshot() -> str:
    try:
        result = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
                '--format=csv,noheader,nounits',
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except Exception as exc:
        return f'nvidia-smi unavailable: {exc}'

print(nvidia_snapshot())

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda device:', torch.cuda.get_device_name(0))
    else:
        print('CellSeg1 can run on CPU, but this training job is intended for NVIDIA CUDA.')
except Exception as exc:
    print('torch import failed:', repr(exc))


In [ ]:
CELLSEG1_REPO = Path(os.getenv('CELLSEG1_REPO', OUTPUT_ROOT / 'cellseg1_repo')).expanduser().resolve()
INSTALL_REQUIREMENTS = os.getenv('CELLSEG1_INSTALL_REQUIREMENTS', '0') == '1'

if not CELLSEG1_REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Nuisal/cellseg1.git', str(CELLSEG1_REPO)], check=True)
else:
    print('Using existing CellSeg1 repo:', CELLSEG1_REPO)

if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CELLSEG1_REPO / 'requirements.txt'), 'PyYAML'], check=True)
else:
    print('Skipping CellSeg1 requirements install. Set CELLSEG1_INSTALL_REQUIREMENTS=1 if this environment is not prepared.')
    try:
        import yaml  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'PyYAML'], check=True)

RUNNER = REPO_ROOT / 'training' / 'run_cellseg1_lora_train.py'
assert RUNNER.exists(), RUNNER
print('CellSeg1 repo:', CELLSEG1_REPO)
print('runner:', RUNNER)


In [ ]:
import urllib.request

SAM_CHECKPOINT = Path(os.getenv('CELLSEG1_SAM_CHECKPOINT', OUTPUT_ROOT / 'sam_backbone' / 'sam_vit_h_4b8939.pth')).expanduser().resolve()
SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'

if SAM_CHECKPOINT.exists() and SAM_CHECKPOINT.stat().st_size > 0:
    print('Using SAM checkpoint:', SAM_CHECKPOINT)
else:
    print('Downloading SAM ViT-H checkpoint to:', SAM_CHECKPOINT)
    urllib.request.urlretrieve(SAM_URL, SAM_CHECKPOINT)
    print('downloaded bytes:', SAM_CHECKPOINT.stat().st_size)


In [ ]:
import yaml

train_ids_raw = os.getenv('CELLSEG1_TRAIN_IDS', '').strip()
if train_ids_raw:
    TRAIN_ID = [int(x) for x in train_ids_raw.split(',') if x.strip()]
else:
    TRAIN_ID = None

EPOCHS = int(os.getenv('CELLSEG1_EPOCHS', '120'))
BATCH = int(os.getenv('CELLSEG1_BATCH', '1'))
GRAD_ACCUM = int(os.getenv('CELLSEG1_GRAD_ACCUM', '32'))
DUPLICATE_DATA = int(os.getenv('CELLSEG1_DUPLICATE_DATA', '64'))
BASE_LR = float(os.getenv('CELLSEG1_BASE_LR', '0.003'))

config = {
    'deterministic': True,
    'allow_tf32_on_matmul': True,
    'allow_tf32_on_cudnn': True,
    'seed': 0,
    'vit_name': 'vit_h',
    'model_path': str(SAM_CHECKPOINT),
    'data_dir': str(RUNTIME_DATASET),
    'result_pth_path': str(RUN_DIR / 'sam_lora_cgh_p2_cell_boundary.pth'),
    'train_image_dir': str(RUNTIME_DATASET / 'images'),
    'train_mask_dir': str(RUNTIME_DATASET / 'masks'),
    'resize_size': [512, 512],
    'patch_size': 0,
    'sam_image_size': 512,
    'train_id': TRAIN_ID,
    'duplicate_data': DUPLICATE_DATA,
    'epoch_max': EPOCHS,
    'batch_size': BATCH,
    'gradient_accumulation_step': GRAD_ACCUM,
    'base_lr': BASE_LR,
    'onecycle_lr_pct_start': 0.3,
    'num_workers': 0,
    'image_encoder_lora_rank': 4,
    'mask_decoder_lora_rank': 4,
    'freeze_image_encoder': True,
    'freeze_prompt_encoder': True,
    'freeze_mask_decoder_transformer': True,
    'freeze_upscaling_cnn': True,
    'freeze_output_hypernetworks_mlps': True,
    'freeze_mask_decoder_mask_tokens': True,
    'freeze_mask_decoder_iou': True,
    'lora_dropout': 0.1,
    'pos_rate': 1.0,
    # CellSeg1 does not natively honor ignore masks, so avoid sampling uncertain/edge regions as hard negatives.
    'neg_rate': 0.0,
    'max_point_num': 30,
    'edge_distance': 20,
    'neg_area_ratio_threshold': 5,
    'neg_area_threshold': 1000,
    'min_cell_area': 100,
    'foreground_sample_area_ratio': 0.2,
    'background_sample_area_ratio': 0.2,
    'foreground_equal_prob': True,
    'background_equal_prob': True,
    'data_augmentation': True,
    'bright_limit': 0.1,
    'contrast_limit': 0.1,
    'bright_prob': 0.5,
    'flip_prob': 0.75,
    'rotate_prob': 0.8,
    'scale_limit': [-0.5, 0.5],
    'crop_prob': 0.5,
    'crop_scale': [0.3, 1.0],
    'crop_ratio': [0.75, 1.3333],
    'ce_loss_weight': 1.0,
    'punish_background_point': False,
    'crop_n_layers': 1,
    'crop_n_points_downscale_factor': 1,
    'points_per_side': 32,
    'points_per_batch': 64,
    'max_mask_region_area_ratio': 0.1,
    'min_mask_region_area': 20,
    'box_nms_thresh': 0.05,
    'crop_nms_thresh': 0.05,
    'pred_iou_thresh': 0.8,
    'stability_score_thresh': 0.6,
    'stability_score_offset': 0.8,
    'track_gpu_memory': False,
}

CONFIG_PATH = RUN_DIR / 'cellseg1_cgh_p2_runtime_config.yaml'
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH.read_text())


In [ ]:
from IPython.display import Markdown, clear_output, display
import shlex

SUMMARY_PATH = RUN_DIR / 'cellseg1_training_summary.json'
LIVE_LOG = RUN_DIR / 'cellseg1_train_live.log'
LIVE_INTERVAL_SECONDS = int(os.getenv('CELLSEG1_LIVE_INTERVAL_SECONDS', '15'))
LOG_TAIL_LINES = int(os.getenv('CELLSEG1_LOG_TAIL_LINES', '80'))

cmd = [
    sys.executable,
    str(RUNNER),
    '--repo', str(CELLSEG1_REPO),
    '--config', str(CONFIG_PATH),
    '--summary', str(SUMMARY_PATH),
]
COMMAND_TEXT = ' '.join(shlex.quote(str(x)) for x in cmd)


def _read_log_tail(log_path: Path, lines: int = LOG_TAIL_LINES) -> str:
    if not log_path.exists():
        return '(training log has not been created yet)'
    text = log_path.read_text(encoding='utf-8', errors='replace').replace('\r', '\n')
    tail = '\n'.join(text.splitlines()[-lines:]).strip()
    return tail or '(training log is empty so far)'


def _format_bytes(path: Path) -> str:
    if not path.exists():
        return 'missing'
    size = path.stat().st_size
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size < 1024 or unit == 'GB':
            return f'{size:.1f} {unit}' if unit != 'B' else f'{size} {unit}'
        size /= 1024


def live_dashboard(run_dir: Path, log_path: Path, started_at: float | None = None) -> None:
    clear_output(wait=True)
    elapsed = ''
    if started_at is not None:
        elapsed = f"\n- Elapsed: {(time.time() - started_at) / 60:.1f} min"

    checkpoint = Path(config['result_pth_path'])
    summary_exists = SUMMARY_PATH.exists()
    log_tail = _read_log_tail(log_path)
    gpu = nvidia_snapshot()

    display(Markdown(f'''
### CellSeg1 Live Training

- Run: `{RUN_NAME}`
- Run dir: `{run_dir}`
- Config: `{CONFIG_PATH}`
- Live log: `{log_path}`{elapsed}
- LoRA checkpoint: `{checkpoint}` ({_format_bytes(checkpoint)})
- Summary JSON: `{SUMMARY_PATH}` ({'exists' if summary_exists else 'not written yet'})
- GPU: `{gpu}`

**Command**
```bash
{COMMAND_TEXT}
```

**Live Log Tail**
```text
{log_tail}
```
'''))


print('Command:')
print(COMMAND_TEXT)
print('GPU before train:', nvidia_snapshot())

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('WANDB_MODE', 'offline')

started = time.time()
with LIVE_LOG.open('w', encoding='utf-8') as log_handle:
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    while proc.poll() is None:
        live_dashboard(RUN_DIR, LIVE_LOG, started)
        time.sleep(LIVE_INTERVAL_SECONDS)
    return_code = proc.wait()

live_dashboard(RUN_DIR, LIVE_LOG, started)
if return_code != 0:
    raise RuntimeError(f'CellSeg1 training failed with return code {return_code}. See {LIVE_LOG}')

print('Training complete')
print('elapsed minutes:', round((time.time() - started) / 60, 2))
print('GPU after train:', nvidia_snapshot())
if SUMMARY_PATH.exists():
    print(SUMMARY_PATH.read_text())
else:
    print(f'Summary file not found: {SUMMARY_PATH}')


In [ ]:
trained_lora = Path(config['result_pth_path'])
deployed_lora = REFERENCE_MODEL_DIR / 'cellseg1_cgh_p2_cell_boundary_lora.pth'
if trained_lora.exists():
    shutil.copy2(trained_lora, deployed_lora)
    deploy_summary = {
        'run_dir': str(RUN_DIR),
        'trained_lora': str(trained_lora),
        'deployed_lora': str(deployed_lora),
        'config': str(CONFIG_PATH),
        'dataset_summary': str(RUN_DIR / 'dataset_runtime_summary.json'),
        'timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    (RUN_DIR / 'cellseg1_deploy_summary.json').write_text(json.dumps(deploy_summary, indent=2), encoding='utf-8')
    print(json.dumps(deploy_summary, indent=2))
else:
    raise FileNotFoundError(f'Trained LoRA checkpoint not found: {trained_lora}')


In [ ]:
# Qualitative check after training: original vs ground truth vs CellSeg1 prediction.
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as IPyImage, display
from PIL import Image

sys.path.insert(0, str(CELLSEG1_REPO))
from data.utils import read_image_to_numpy, resize_image
from predict import predict_images

trained_lora = Path(config['result_pth_path'])
assert trained_lora.exists(), trained_lora

PREDICT_LIMIT = int(os.getenv('CELLSEG1_PREDICT_LIMIT', '4'))
PRED_IOU = float(config['pred_iou_thresh'])
PRED_STABILITY = float(config['stability_score_thresh'])
COMPARE_DIR = RUN_DIR / f'comparison_original_gt_pred_iou{PRED_IOU:.2f}_stab{PRED_STABILITY:.2f}'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

GT_COLOR = np.array([0, 180, 120], dtype=np.uint8)      # green in RGB
PRED_COLOR = np.array([255, 200, 0], dtype=np.uint8)    # amber in RGB

selected_images = image_files[:PREDICT_LIMIT]
images = [resize_image(read_image_to_numpy(p), config['resize_size']) for p in selected_images]
pred_masks = predict_images(config, images)

PRED_MASK_DIR = RUN_DIR / 'pred_masks'
PRED_MASK_DIR.mkdir(exist_ok=True)


def resize_mask_to_image(mask: np.ndarray, img_rgb: np.ndarray) -> np.ndarray:
    target_h, target_w = img_rgb.shape[:2]
    if mask.shape[:2] == (target_h, target_w):
        return mask
    return cv2.resize(mask.astype(np.uint16), (target_w, target_h), interpolation=cv2.INTER_NEAREST)


def mask_boundaries(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask)
    boundary = np.zeros(mask.shape, dtype=bool)
    boundary[1:, :] |= mask[1:, :] != mask[:-1, :]
    boundary[:-1, :] |= mask[1:, :] != mask[:-1, :]
    boundary[:, 1:] |= mask[:, 1:] != mask[:, :-1]
    boundary[:, :-1] |= mask[:, 1:] != mask[:, :-1]
    return boundary & (mask > 0)


def draw_instance_mask(img_rgb: np.ndarray, mask: np.ndarray, color: np.ndarray, alpha: float = 0.30) -> np.ndarray:
    mask = resize_mask_to_image(mask, img_rgb)
    out = img_rgb.copy()
    overlay = img_rgb.copy()
    foreground = mask > 0
    overlay[foreground] = color

    boundaries = mask_boundaries(mask)
    out[boundaries] = color

    # Thicker contours make sparse CellSeg1 instance boundaries easier to review.
    for label in sorted(int(v) for v in np.unique(mask) if int(v) != 0):
        mask_bin = (mask == label).astype(np.uint8)
        contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, tuple(map(int, color)), 2)

    return cv2.addWeighted(overlay, alpha, out, 1 - alpha, 0)


saved = []
for image_path, image, pred_mask in zip(selected_images, images, pred_masks):
    tile_id = image_path.stem
    gt_path = RUNTIME_DATASET / 'masks' / image_path.name
    gt_mask = np.asarray(Image.open(gt_path))
    gt_mask = resize_mask_to_image(gt_mask, image)
    pred_mask = resize_mask_to_image(pred_mask, image)

    pred_path = PRED_MASK_DIR / image_path.name
    Image.fromarray(pred_mask.astype(np.uint16)).save(pred_path)

    gt = draw_instance_mask(image, gt_mask, GT_COLOR)
    pred = draw_instance_mask(image, pred_mask, PRED_COLOR)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    panels = [image, gt, pred]
    titles = [
        f'{tile_id}: original',
        f'ground truth instances n={int(gt_mask.max())}',
        f'CellSeg1 prediction n={int(pred_mask.max())}, iou={PRED_IOU:.2f}, stability={PRED_STABILITY:.2f}',
    ]
    for ax, im, title in zip(axes, panels, titles):
        ax.imshow(im)
        ax.set_title(title)
        ax.axis('off')

    handles = [
        plt.Line2D([0], [0], color=GT_COLOR / 255, lw=4, label='ground truth cell boundary'),
        plt.Line2D([0], [0], color=PRED_COLOR / 255, lw=4, label='CellSeg1 prediction'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=len(handles))
    fig.tight_layout(rect=[0, 0.06, 1, 1])

    out_path = COMPARE_DIR / f'{tile_id}_original_gt_pred.png'
    fig.savefig(out_path, dpi=180)
    plt.close(fig)
    saved.append(out_path)
    display(IPyImage(filename=str(out_path)))

print('Saved comparisons to:', COMPARE_DIR)
print('Files:')
for path in saved:
    print(path)
